# Aggregasi & Business Insight per Gerai
Perhitungan frekuensi aspek dan persentase komplain (sentimen negatif) di tingkat gerai/cabang untuk laporan visualisasi.

In [ ]:
# Setup environment jika dijalankan di Google Colab / Kaggle
import os
import sys

if 'google.colab' in sys.modules:
    print("Colab detected. Pastikan dataset 'reviews_with_aspect_sentiment.csv' sudah diupload.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
dataset_path = '../data/reviews_with_aspect_sentiment.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'reviews_with_aspect_sentiment.csv'

df = pd.read_csv(dataset_path)
print(f"Total data: {len(df)} baris")

## 2. Hitung Agregasi Aspek per Gerai

In [ ]:
aspect_cols = ['aspek_rasa', 'aspek_harga', 'aspek_pelayanan', 'aspek_kecepatan', 'aspek_kebersihan', 'aspek_stok', 'aspek_aplikasi']

# Ekstrak rating numerik jika belum tersedia
if 'rating_num' not in df.columns and 'rating' in df.columns:
    df['rating_num'] = df['rating'].str.extract('(\d+)').astype(float)
elif 'rating_num' not in df.columns:
    df['rating_num'] = np.nan

# 1. Agregasi review dan rating
gerai_base = df.groupby('nama_gerai').agg(
    total_review=('ulasan', 'count'),
    avg_rating=('rating_num', 'mean')
).reset_index()

# 2. Frekuensi kemunculan aspek
gerai_aspects = df.groupby('nama_gerai')[aspect_cols].sum().reset_index()

# 3. Rasio sentimen negatif per aspek
def calculate_negative_ratio(group):
    ratios = {}
    for aspect in aspect_cols:
        sentiment_col = aspect.replace('aspek_', 'sentimen_')
        aspect_mentions = group[aspect].sum()
        neg_mentions = (group[sentiment_col] == 'negative').sum()
        ratios[aspect.replace('aspek_', 'neg_ratio_')] = neg_mentions / aspect_mentions if aspect_mentions > 0 else 0.0
    return pd.Series(ratios)

gerai_neg_ratios = df.groupby('nama_gerai').apply(calculate_negative_ratio).reset_index()

aggregated_data = pd.merge(gerai_base, gerai_aspects, on='nama_gerai')
aggregated_data = pd.merge(aggregated_data, gerai_neg_ratios, on='nama_gerai')

# Flag outlet dengan review < 20 (low confidence)
aggregated_data['confidence_level'] = aggregated_data['total_review'].apply(lambda x: 'High' if x >= 20 else 'Low')
aggregated_data

## 3. Identifikasi Cabang Bermasalah

In [ ]:
high_confidence_df = aggregated_data[aggregated_data['confidence_level'] == 'High']

for col in [c for c in aggregated_data.columns if c.startswith('neg_ratio_')]:
    aspect_clean = col.replace('neg_ratio_', '').upper()
    print(f"\n3 Cabang dengan Masalah {aspect_clean} Tertinggi (Rasio Negatif):")
    top_issues = high_confidence_df.sort_values(by=col, ascending=False).head(3)
    for _, row in top_issues.iterrows():
        print(f"- {row['nama_gerai']}: {row[col] * 100:.2f}% review negatif (Total sebutan: {int(row['aspek_' + col.replace('neg_ratio_', '')])})")

## 4. Visualisasi Heatmap Komplain

In [ ]:
heatmap_data = high_confidence_df.set_index('nama_gerai')[[c for c in aggregated_data.columns if c.startswith('neg_ratio_')]]
heatmap_data.columns = [c.replace('neg_ratio_', '').upper() for c in heatmap_data.columns]

plt.figure(figsize=(14, 8))
sns.heatmap(heatmap_data * 100, annot=True, cmap='OrRd', fmt='.1f', linewidths=.5)
plt.title('Persentase Sentimen Negatif Per Aspek Per Gerai Kopi Kenangan (%)')
plt.xlabel('Aspek')
plt.ylabel('Nama Gerai')
plt.tight_layout()
plt.show()

## 5. Simpan Hasil Agregasi

In [ ]:
os.makedirs('../data', exist_ok=True)

df.to_csv('../data/aspect_analysis_gmaps.csv', index=False)
aggregated_data.to_csv('../data/aggregated_insight_per_gerai.csv', index=False)
print("Data agregasi berhasil disimpan ke modeling/data/")